# NVIDIA Modelcard Scraper Verification

This notebook verifies the `NvidiaModelcardScraper` + `infer_tool_support()` logic against NVIDIA Build modelcards.

**What it does**
- Fetches modelcard HTML with browser-like headers (HTTP/2, redirects)
- Extracts title/description/visible text
- Extracts SPA JSON blobs (`__NEXT_DATA__`, `application/ld+json`, `application/json`)
- Runs a recursive search for strong keywords: `function calling`, `tool calling`, `tool_calls`, etc.
- Prints evidence snippets and a boolean/unknown tool-support inference.

> Run this from your project root (or set `PROJECT_ROOT` below).


In [1]:
# If needed, install deps (run once)
%pip install ipykernel

/home/pruthvi/projects/h-ai/HFCL-FastMCP-server-httpx-tools/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, sys, logging, textwrap

# ---- CONFIG ----
# Set this to your repo root path if running the notebook elsewhere.
PROJECT_ROOT = os.getenv("PROJECT_ROOT", os.getcwd())

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

logging.basicConfig(level=logging.INFO)
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /home/pruthvi/projects/h-ai/HFCL-FastMCP-server-httpx-tools


In [2]:
# Import your scraper module
# Adjust import path if your package layout differs.
from src.agent_service.nvidia_modelcard import NvidiaModelcardScraper, infer_tool_support


In [3]:
scraper = NvidiaModelcardScraper(http2=True, timeout_s=20.0, retries=3)

models = [
    "deepseek-ai/deepseek-r1",
    "deepseek-ai/deepseek-r1-0528",
]


In [4]:
# Jupyter supports top-level `await` (IPython). If not, wrap in asyncio.run().
import time

async def check_model(mid: str):
    print("\n" + "="*110)
    print("MODEL:", mid)
    res = await scraper.scrape(mid)
    tool_supported, evidence = infer_tool_support(res)

    print(f"URL:         {res.url}")
    print(f"HTTP:        {res.status_code}")
    print(f"TITLE:       {res.title}")
    if res.description:
        print(f"DESCRIPTION: {res.description}")

    print("\nTOOL SUPPORT INFERRED:", tool_supported)
    if evidence:
        print("\nEVIDENCE SNIPPETS:")
        for i, ev in enumerate(evidence[:12], 1):
            print(f"{i:02d}. {ev}")
    else:
        print("\nEVIDENCE: (none)")

    # quick peek at visible text size
    print("\nVISIBLE TEXT LEN:", len(res.visible_text or ""))
    print("JSON BLOBS:", len(res.json_blobs))

for mid in models:
    await check_model(mid)



MODEL: deepseek-ai/deepseek-r1


INFO:httpx:HTTP Request: GET https://build.nvidia.com/deepseek-ai/deepseek-r1/modelcard "HTTP/2 200 OK"


URL:         https://build.nvidia.com/deepseek-ai/deepseek-r1/modelcard
HTTP:        200
TITLE:       deepseek-r1 Model by Deepseek-ai | NVIDIA NIM
DESCRIPTION: State-of-the-art, high-efficiency LLM excelling in reasoning, math, and coding.

TOOL SUPPORT INFERRED: None

EVIDENCE: (none)

VISIBLE TEXT LEN: 0
JSON BLOBS: 0

MODEL: deepseek-ai/deepseek-r1-0528


INFO:httpx:HTTP Request: GET https://build.nvidia.com/deepseek-ai/deepseek-r1-0528/modelcard "HTTP/2 200 OK"


URL:         https://build.nvidia.com/deepseek-ai/deepseek-r1-0528/modelcard
HTTP:        200
TITLE:       deepseek-r1-0528 Model by Deepseek-ai | NVIDIA NIM
DESCRIPTION: Updated version of DeepSeek-R1 with enhanced reasoning, coding, math, and reduced hallucination.

TOOL SUPPORT INFERRED: None

EVIDENCE: (none)

VISIBLE TEXT LEN: 0
JSON BLOBS: 0


In [5]:
# Optional: quick regression assertion (edit expected values as you observe in prod)
expected = {
    "deepseek-ai/deepseek-r1": False,          # you said: NO tools on NVIDIA
    "deepseek-ai/deepseek-r1-0528": True,      # you said: YES tools on NVIDIA
}

# NOTE: infer_tool_support can return None if the modelcard doesn't explicitly mention tools.
# If that happens, treat it as "unknown" and don't fail hard.
def assert_expected(mid: str, inferred):
    exp = expected.get(mid)
    if exp is None:
        return
    if inferred is None:
        print(f"[WARN] {mid}: inferred=None (unknown); expected={exp}. Modelcard text may not mention tools explicitly.")
        return
    assert inferred == exp, f"{mid}: inferred={inferred} expected={exp}"

print("Done.")


Done.
